<h4>定义 Определение</h4>

In [104]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from math import floor
import csv

<h4>在此定义输入维度和目标维度</h4>
需要将测试集里面的Very Low转化成very_low

In [105]:
type = {"very_low":0,"Low":1,"Middle":2,"High":3}#将类型变为数字 Преобразование типов в числа
name = ["very_low","Low","Middle","High"]#将数字变为类型 Преобразование чисел в типы
num_Target = len(name)
dim_Input = 5

<h4>读取文件并把向量数字化 Чтение файла и векторизация данных</h4>
需要依据情况修改sep

In [106]:
file = 'train.csv'
df = pd.read_csv(file, encoding='utf-8',sep=',')
array = df.to_numpy()
for j in range(len(array)):
    array[j][dim_Input] = type[array[j][dim_Input]]

file = 'test.csv'
df = pd.read_csv(file, encoding='utf-8',sep=',')
arrayTest = df.to_numpy()
for j in range(len(arrayTest)):
    arrayTest[j][dim_Input] = type[arrayTest[j][dim_Input]]

<h4>提取初始以及目标向量 Извлечение начальных и целевых векторов</h4>

In [107]:
X = array[:, :dim_Input]
Y = array[:, dim_Input]
X_test = arrayTest[:, :dim_Input]
Y_test = arrayTest[:, dim_Input]
num_INP = len(array)
probability = []#将类型转化为概率
num_INP = len(array)
matrixTarget = np.eye(num_Target)
for i in range(num_INP):
    probability.append(matrixTarget[Y[i]])

#min-max归一化避免非线性模型数值溢出
X_min = X.min(axis=0)
X_max = X.max(axis=0)
X = (X - X_min) / (X_max - X_min + 1e-8)

类跃迁函数Sigmoid

In [108]:
def sigmoid(inpLst):
    return 1/(1+np.exp(-inpLst))
def ReLU(x_):
    return x_ if x_ > 0 else 0
def np_ReLU(x_):
    return np.maximum(0,x_)
def softmax(x_):
    return np.exp(x_ - np.max(x_, keepdims=True)) / np.sum(np.exp(x_ - np.max(x_, keepdims=True)), keepdims=True)
def numerical_gradient(f, x, epsilon=1e-4):
    grad = np.zeros_like(x)
    # 使用 ndenumerate 遍历任意维度的数组，绝对不会越界
    for idx_tuple, _ in np.ndenumerate(x):
        old_val = x[idx_tuple]
        x[idx_tuple] = old_val + epsilon
        loss_plus = f()
        x[idx_tuple] = old_val - epsilon
        loss_minus = f()
        grad[idx_tuple] = (loss_plus - loss_minus) / (2 * epsilon)
        x[idx_tuple] = old_val
    return grad

深度学习类

In [109]:
class neuronCell:#神经元（单层）
    def __init__(self,inpDim_,optDim_):
        self.inputDim=inpDim_
        self.outputDim=optDim_
        self.weights = np.random.randn(inpDim_, optDim_) * np.sqrt(2.0 / inpDim_)
        self.b = np.zeros((1, optDim_))
        #参数
        self.x = None   #输入
        self.y = None   #矩阵乘法结果
        self.z = None   #输出
        self.dx = None  #梯度输入
        self.dw = None  #梯度权重
        self.db = None  #梯度偏置

    def vowart(self):
        raise NotImplementedError(f"类{self.__class__.__name__}没有前向传播函数")

    def backwart(self):
        raise NotImplementedError(f"类{self.__class__.__name__}没有后向传播函数")
    
    def meanSquareError(self, target_,average=True):#均方差/平方和误差
        diff = self.z - target_
        N = self.z.shape[0]
        if average:
            loss = np.sum(diff**2) / N
            dout = 2 * diff / N
        else:
            loss = np.sum(diff**2)
            dout = 2 * diff
        return loss, dout
    
    def clear(self):
        self.x = None
        self.z = None
        self.dx = None
        self.dw = None
        self.db = None

class linCell(neuronCell):#线性神经元
    def __init__(self,inpDim_,optDim_):
        super().__init__(inpDim_,optDim_)

    def vowart(self, inp_):
        inp_ = np.array(inp_)
        if inp_.ndim == 1:
            if inp_.size % self.inputDim != 0:
                raise ValueError(f"输入元素个数 {inp_.size} 无法整除输入维度 {self.inputDim}")
            inp_ = inp_.reshape(-1, self.inputDim)
        self.x = inp_
        self.z = np.dot(inp_, self.weights) +self.b
        #print(f"输入形状: {inp_.shape}")
        #print(f"z形状: {self.z.shape}")
        return self.z

    def backwart(self, inp_):
        self.dw = np.dot(self.x.T, inp_)
        self.db = np.sum(inp_, axis=0, keepdims=True)
        self.dx = np.dot(inp_, self.weights.T)
        #print(f"输入形状: {inp_.shape}")
        #print(f"z形状: {self.dx.shape}")
        return self.dx
    
    def meanSquareError(self, target_,average=True):
        if target_.ndim == 1:
            target_ = target_.reshape(-1, 1)
        return super().meanSquareError(target_,average)
    
class ActivationCell:#激活层
    def __init__(self):
        self.x = None
        self.z = None
        self.dx = None

    def vowart(self, inp_):
        raise NotImplementedError

    def backwart(self, dout):
        raise NotImplementedError

    def clear(self):
        self.x = None
        self.z = None
        self.dx = None

class ReLUCell(ActivationCell):#ReLU激活层
    def vowart(self, inp_):
        self.x = inp_
        self.z = np_ReLU(inp_)
        return self.z

    def backwart(self, dout):
        self.dx = dout * (self.x > 0)
        return self.dx
    
class sigmoidCell(ActivationCell):#sigmoid激活层
    def vowart(self, inp_):
        self.x = inp_
        self.z = sigmoid(inp_)
        return self.z

    def backwart(self, dout):
        local_grad = self.z * (1 - self.z)
        self.dx = dout * local_grad
        return self.dx

class softmaxCell(ActivationCell):#softmax激活层
    def vowart(self, inp_):
        self.x = inp_
        self.z = softmax(inp_)
        return self.z

    def backwart(self, dout):
        p = self.z
        # 计算 sum_j dout_j * p_j，即逐样本的加权和
        sum_dout_p = np.sum(dout * p, axis=1, keepdims=True)
        self.dx = p * (dout - sum_dout_p)
        return self.dx

LAYER_REGISTRY = {
    "linear": linCell,  
    "relu": ReLUCell,
    "sigmoid": sigmoidCell,
    "softmax": softmaxCell
}
class neuralNet:#神经网络   最后一层必须是线性层
    def __init__(self,typeLayers_,inpDims_,optDims_):
        self.layers=[]
        indexx=0
        for i in range(len(typeLayers_)):
            layerName = typeLayers_[i].lower()
            if layerName == "linear":
                self.layers.append(LAYER_REGISTRY[layerName](inpDims_[indexx],optDims_[indexx]))
                indexx+=1
            else:
                self.layers.append(LAYER_REGISTRY[layerName]())
            
    def vowart(self, inp_):
        inp_ = np.array(inp_)
        for layer in self.layers:
            inp_ = layer.vowart(np.array(inp_))
        return inp_

    def backwart(self, inp_):
        for layer in reversed(self.layers):
            inp_ = layer.backwart(np.array(inp_))
        return inp_

    def clear_cache(self):
        for layer in self.layers:
            layer.clear_cache()

    def train(self,inp_, target_,learningRate):#训练逻辑：前向传播，计算损失和输入梯度，反向传播，梯度下降
        predict = self.vowart(inp_)
        loss, dout = self.layers[-1].meanSquareError(target_, average=True)
        self.backwart(dout)
        for layer in self.layers:
            if hasattr(layer, 'weights') and hasattr(layer, 'dw'):
                layer.weights -= learningRate * layer.dw
                layer.b -= learningRate * layer.db
        return loss

    def meanSquareError(self,inp_, target_,average=True):#均方差/平方和误差
        tmp = self.vowart(inp_)
        diff = tmp - target_
        N = tmp.shape[0]
        if average:
            loss = np.sum(diff**2) / N
        else:
            loss = np.sum(diff**2)
        return loss

<h4>训练及模型 Обучение и модель</h4>

In [110]:
epsilon = 0.03#定义容许误差 Определение допустимой погрешности
maxTime = int(1E5)
alpha = 1E-2
numCell=16
neural=neuralNet(["linEar","relu","liNEaR","relu","liNEaR","relu","liNEaR"],[dim_Input,numCell,numCell,numCell],[numCell,numCell,numCell,num_Target])
X = X.astype(np.float64)
X_test = X_test.astype(np.float64)

print(f"训练前均方差: {neural.meanSquareError(X,np.array(probability))}")
for i in range(maxTime):
    lo = neural.train(X,np.array(probability),alpha)
    #print(lo)
    if(lo<epsilon):
        break
print(f"训练后均方差: {neural.meanSquareError(X,np.array(probability))}")
probability_predict = neural.vowart(X)
probability_test = neural.vowart(X_test)

训练前均方差: 2.489482104740528
训练后均方差: 0.030000178345003493


In [111]:
howGood = []
howGoodTest = []
def greatest(inp_):
    return np.argmax(inp_)
for i in probability_predict:
    howGood.append(np.argmax(i))
for i in probability_test:
    howGoodTest.append(np.argmax(i))
err=np.sum(howGood!=Y)
err2=np.sum(howGoodTest!=Y_test)
print(f"训练集错误数: {err}")
print(f"测试集错误数: {err2}")
print(f"训练集错误率: {err*100/len(howGood):.2f}%")
print(f"测试集错误率: {err2*100/len(howGoodTest):.2f}%")

训练集错误数: 4
测试集错误数: 5
训练集错误率: 1.55%
测试集错误率: 3.45%
